# 记忆管理

大多数 LLM 都有最大支持的上下文窗口，模型不是无限容量的。随着对话变长，你的 `checkpoints.db` 会越来越大，发给模型的 Token 也会越来越多（意味着越来越贵、越来越慢）。

## 常见模式（如何选择管理记忆的方式？）

| **模式**          | **你的真实意图**                           | **结果**                                   | **推荐场景**                                   |
| ----------------- | ------------------------------------------ | ------------------------------------------ | ---------------------------------------------- |
| **Trim Messages** | “我只想让模型关注最近的事，省点钱。”       | 数据库里**保留**全量，发送时**只发**部分。 | **默认首选。** 既省钱，又能随时查历史。        |
| **RemoveMessage** | “这段对话太尴尬了/出错了，彻底抹掉。”      | 数据库里**彻底删除**。                     | 隐私保护、错误修正、手动清理。                 |
| **Summarization** | “以前的事很重要，但我不想发几千字给模型。” | 数据库里**删除旧消息**，存入一条**总结**。 | 极长期的复杂对话（如写小说、长期伴侣机器人）。 |



### Trim Messages 修剪消息

**Trim Messages** 就像是给 AI 戴上一个“滑动滤镜”。它不是真的从数据库里永久删除（那是 `RemoveMessage` 干的事），而是在**发送给模型那一刻**，决定只截取其中一段。

在实际开发中，不需要从零写逻辑，`LangChain` 提供了一个内置工具 `trim_messages`。它之所以是“模式”，是因为它有一套成熟的选择策略。

#### 怎么切？

你会根据以下三个场景来决定怎么“切”：

#### 场景 A：按数量切 (Last N Messages)

- **策略：** 只给模型看最近的 10 条消息。
- **理由：** 简单粗暴，适用于大多数日常闲聊。

#### 场景 B：按 Token 数切 (Max Tokens) —— **最常用**

- **策略：** 不管有几条消息，只要总 Token 数超过了 4096，就从旧的开始扔掉，直到剩下 4096 以内。
- **理由：** 这是为了**省钱**和**防崩溃**。因为模型收费是按 Token 算的，如果你不修剪，最后一次对话可能要花掉你几块钱。

#### 场景 C：保留系统提示词 (Protect System Message)

- **策略：** 无论怎么切，第一条 `SystemMessage`（规定 AI 人设的那条）必须留下。
- **理由：** 如果把人设切掉了，AI 就会“失忆”，不知道自己是谁。

#### 代码示例



In [6]:
try:
    import transformers
    print("Transformers 导入成功，版本：", transformers.__version__)
except ImportError:
    print("代码运行时确实找不到 transformers")

e:\code\LangChainDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Transformers 导入成功，版本： 5.5.1


In [ ]:
import operator
from typing import Annotated, Any, TypedDict, cast

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    HumanMessage,
    AnyMessage,
    SystemMessage,
    trim_messages,
)
from langgraph.checkpoint.sqlite import SqliteSaver


class CustomAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    user_id: str
    preferences: dict


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

# 创建修剪器 Trimmer
trimmer = trim_messages(
    max_tokens=600,  # 限制总长度（根据你的模型窗口调整）
    strategy="last",  # 保留最近的消息
    token_counter="approximate", # 传入字符串，内置的 approximate，进行单词/字符估算
    include_system=True,  # ！！！非常重要：无论怎么剪，不要丢掉系统人设
    start_on="human",  # 剪切后的第一条必须是人类消息，防止 AI 逻辑混乱
)

memory_context = SqliteSaver.from_conn_string("checkpoints.db")

with memory_context as memory:
    agent = create_agent(
        model=model,
        tools=[],
        state_schema=cast(Any, CustomAgentState),
        checkpointer=memory,
    )
    config = {"configurable": {"thread_id": "webster_long_chat"}}
    input_data = {
        "messages": [
            SystemMessage(content="你是 Webster 的贴心助手"),  # 系统人设
            HumanMessage(content="这是第 1 条废话..."),
            # ... 假设这里有 100 条消息 ...
            HumanMessage(content="这是最后一条关键指令：我的幸运数字是 7"),
        ],
        "user_id": "user_123",
        "preferences": {"theme": "dark"},
    }
    # 在 invoke 之前手动执行修剪
    input_data["messages"] = trimmer.invoke(input_data["messages"])
    
    print(f"--- 修剪后的消息条数: {len(input_data['messages'])} ---")
    result = agent.invoke(cast(Any, input_data), config)
    print(f"AI 回复: {result['messages'][-1].content}")


--- 修剪后的消息条数: 3 ---
AI 回复: 了解，你的幸运数字是7。希望这个数字能给你带来好运和快乐！如果有其他事情需要帮助，请随时告诉我。



> ### 为什么要这么写？
>
> 1. **`include_system=True` 是灵魂：** 如果你不设置这个，修剪器可能会把最开头的 `SystemMessage` 剪掉。结果就是：你的 AI 会突然忘记自己是你的助手，变成一个“失忆”的原始模型。
>
> 2. **`token_counter=model` 的精准度：** 不同的模型对同一个句子计算出的 Token 数是不一样的。让 `qwen` 亲自数它自己的 Token，能保证修剪得恰到好处，既不浪费空间，也不会溢出。
>
>    > ##### 坑
>    >
>    > 在 LangChain 的设计中，`token_counter=model` 有一个前提：**该模型对象必须实现了 `get_num_tokens` 方法**。
>    >
>    > 不幸的是，`init_chat_model` 返回的对象（尤其是对接 Ollama 时），有时并不会自动携带计算 Token 的逻辑。当 `trim_messages` 发现传入的 `model` 没法告诉它具体的 Token 数时，它会触发**“降级策略”**：
>    >
>    > 1. 它想：“既然没告诉我怎么算，那我就默认按 GPT-2 的分词方式算吧。”
>    > 2. 为了按 GPT-2 算，它就必须去加载本地的 `transformers` 库。
>    > 3. 结果你本地没能成功加载，就报了那个 `ImportError`。
>
>    传入`token_counter="gpt-4o"`,运行之后报错信息推荐使用`token_counter="approximate"`,解决这个坑。
>
> 3. **为什么不改 `agent` 内部，而是手动 `trimmer.invoke`？** 虽然 `create_agent` 是个黑盒，但通过在 `invoke` 前处理 `input_data`，你可以完全控制送入“大脑”的内容。这正是 **Common Pattern** 的精髓：**在数据流动过程中拦截并优化。**

> ### 使用管道替代手动调用
>
> 

In [ ]:
# 将模型包装在一个包含修剪逻辑的管道中
# 这样你以后直接调用 agent，它内部就会自动修剪，不需要你在外面手动传
chain = trimmer | model

agent = create_agent(
    model=chain,  # 这里的 model 变成了带修剪功能的 chain
    tools=[],
    state_schema=cast(Any, CustomAgentState),
    checkpointer=memory,
)

### Remove Messages

它不仅仅是让模型“看不见”，而是从你的数据库（`checkpoints.db`）和状态（`State`）中**永久、物理性地删除**某段记忆。

#### 1. 为什么需要 RemoveMessage？

虽然 Trim 能省 Token，但有些场景它处理不了：

- **用户反悔了：** “擦除刚才关于我银行卡密码的那段对话。”
- **错误纠正：** Agent 刚才产生了一个幻觉，回答错了，你希望从历史里把那段“黑历史”删掉，以免误导后面的对话。
- **状态重置：** 虽然 thread_id 没变，但你想清空所有聊天记录重新开始。

#### 2. RemoveMessage 的工作原理

在 LangGraph 中，消息列表是**只增不减**的（因为使用了 `operator.add`）。你不能直接操作列表去 `pop()` 或 `remove()`。

你必须向 `messages` 列表发送一个特殊的**占位符对象**：`RemoveMessage(id="xxx")`。当 LangChain 看到这个对象时，它会去数据库里找对应 `id` 的消息并将其移除。

#### 3. 代码示例：如何精准删除一条消息

为了让你看清效果，我们手动获取消息的 `id`，然后执行删除。

#### 4. RemoveMessage 的两个核心特征

##### A. 必须指定 ID

每一条消息在存入 `SqliteSaver` 时，都会被自动分配一个 UUID（除非你手动指定）。要删除消息，**ID 是唯一的凭证**。

##### B. 它是“逻辑相减”

因为你在 `State` 里定义了 `Annotated[list, operator.add]`，LangGraph 内部有一个特殊的逻辑：

- 当它遇到 `HumanMessage` -> **加**进列表。
- 当它遇到 `RemoveMessage` -> 从列表中**减**去对应 ID 的项。

#### 5. 常见实战模式：清空所有历史

如果你想给用户提供一个“清空聊天记录”的按钮，你可以这样做：

```python
def clear_all_history(agent, config):
    # 获取当前所有消息
    current_state = agent.get_state(config)
    all_messages = current_state.values.get("messages", [])
    
    # 为每一条消息创建一个 RemoveMessage
    delete_cmds = [RemoveMessage(id=m.id) for m in all_messages if m.id]
    
    # 一次性发送，清空数据库
    if delete_cmds:
        agent.invoke({"messages": delete_cmds}, config)
        print("所有历史已物理删除。")
```


In [13]:

from langchain_core.messages import RemoveMessage


memory_context = SqliteSaver.from_conn_string("checkpoints.db")

with memory_context as memory:
    agent = create_agent(
        model=model,
        tools=[],
        state_schema=cast(Any, CustomAgentState),
        checkpointer=memory,
    )
    config = {"configurable": {"thread_id": "webster_long_chat"}}
    # 1. 先随便发一句话
    result = agent.invoke(
        {
            "messages":[
                HumanMessage(content="记住我的暗号是：芝麻开门")
            ]
        },
        config
    )
    # 2. 找到刚才那条消息的 ID
    # result["messages"] 包含了所有历史，我们拿最后一条 human 消息的 id
    last_message_id = result["messages"][-2].id #倒数第二条通常是HumanMessage
    print(f"准备删除的消息ID：{last_message_id}")

    # 3. 执行删除操作
    # 关键：发送一个 RemoveMessage 对象
    agent.invoke({"messages": [cast(Any, RemoveMessage(id=last_message_id))]}, config)

    # 4. 验证：再次询问，看它还记得暗号吗
    final_result = agent.invoke({"messages": [HumanMessage(content="我的暗号是什么？")]}, config)
    print(f"AI 回复: {final_result['messages'][-1].content}")

准备删除的消息ID：None


TypeError: Received unsupported message type for Ollama.

> ### 1. 为什么 `last_message_id` 是 `None`？
>
> 在 LangChain 中，如果你手动创建一条消息 `HumanMessage(content="...")`，它是没有 ID 的。只有当消息经过 **Checkpointer（SqliteSaver）** 存储并返回后，系统才会给它分配一个 UUID。
>
> 之所以你拿到了 `None`，通常是因为在 `create_agent` 的某些版本或 Ollama 的响应流中，返回的消息对象还没有被完全“持久化”。
>
> ### 2. 为什么会报 `TypeError: Received unsupported message type for Ollama`？
>
> 这个报错是因为 **Ollama 不认识 `RemoveMessage`**。
>
> - **真相：** `RemoveMessage` 并不是发给 AI 模型的，它是发给 **LangGraph 控制中心** 的指令。
> - **报错原因：** 你使用了 `agent.invoke(...)`。这个方法会试图把整个 `messages` 列表（包括你刚加进去的 `RemoveMessage`）塞给 Ollama。Ollama 一看：“这是啥格式？我不认识！”于是就报错了。
>
> 要正确执行删除，我们不能直接调 `agent.invoke` 让 AI 去处理删除指令，而是应该**更新状态（Update State）**。这样指令会直接作用于数据库，而不会惊动 AI 模型。
>
> ###  核心知识点：`invoke` vs `update_state`
>
> - **`agent.invoke`**：相当于**“对话”**。你说话 -> 图运行 -> 经过 AI 模型 -> AI 回话。AI 如果不认识 `RemoveMessage` 就会崩。
> - **`agent.update_state`**：相当于**“修改存档”**。你直接翻开 `checkpoints.db`，把那一页撕掉。这个过程**不经过 AI 模型**，所以非常安全、快速。
>
> ### 总结
>
> 1. **拿 ID**：通过 `agent.get_state(config)` 去拿最稳妥。
> 2. **删消息**：用 `agent.update_state` 绕过模型。
>
> **试着换成 `update_state` 方法，报错应该就烟消云散了，AI 也会真的把“芝麻开门”忘得一干二净！**


In [22]:
import uuid
from langchain_core.messages import RemoveMessage, HumanMessage

# 1. 换一个新的 thread_id，保证环境绝对纯净
config = {"configurable": {"thread_id": "webster_final_test_999"}}
memory_context = SqliteSaver.from_conn_string("checkpoints.db")

with memory_context as memory:
    agent = create_agent(model=model, tools=[], state_schema=CustomAgentState, checkpointer=memory)

    # --- 第一步：存入暗号 ---
    msg_id = str(uuid.uuid4())
    print(f"设定 ID: {msg_id}")
    agent.invoke({"messages": [HumanMessage(content="暗号是芝麻开门", id=msg_id)]}, config)

    # --- 第二步：手术删除 ---
    # 关键点：我们不仅删除，还要确认删除后的状态
    agent.update_state(config, {"messages": [RemoveMessage(id=msg_id)]})
    
    # 打印出来检查一下：现在的历史记录里到底还有没有那条消息？
    current_messages = agent.get_state(config).values["messages"]
    print(f"当前历史消息条数（不应包含被删的那条）: {len(current_messages)}")
    
    # 如果你在这里看到 RemoveMessage 对象，说明 reducer 没起作用
    print("当前历史列表内容类型:", [type(m) for m in current_messages])

    # --- 第三步：验证 ---
    # 如果上面的类型列表里还有 RemoveMessage，我们要手动过滤它（这是一个 Bug 规避方案）
    print("--- 开始验证 ---")
    try:
        final_result = agent.invoke({"messages": [HumanMessage(content="我的暗号是什么？")]}, config)
        print(f"AI 回复: {final_result['messages'][-1].content}")
    except Exception as e:
        print(f"依然报错，原因是: {e}")

设定 ID: 65cc0323-707d-44dd-b16d-ce3d6a71ad4d
当前历史消息条数（不应包含被删的那条）: 3
当前历史列表内容类型: [<class 'langchain_core.messages.human.HumanMessage'>, <class 'langchain_core.messages.ai.AIMessage'>, <class 'langchain_core.messages.modifier.RemoveMessage'>]
--- 开始验证 ---
依然报错，原因是: Received unsupported message type for Ollama.


代码依然报 Unsupported message type，那结论只有一个：使用的 create_agent 底层的 Reducer（合并逻辑）不认识 RemoveMessage。

这是一个很坑的兼容性问题。如果发生了这种情况，说明 create_agent 默认的 State 处理方式太简单了，它把 RemoveMessage 当成普通消息存着，却不知道怎么用它去“减掉”之前的消息。

TODO 除了手动删库,暂时我还没好办法.
直接操作 SQLite 数据库清空这个 thread 的所有记录：

In [28]:
# 这一招叫“掀桌子”，直接重置这个人的记忆，不走 RemoveMessage 的弯路
def hard_reset_memory(conn_string, thread_id):
    import sqlite3
    conn = sqlite3.connect(conn_string)
    cursor = conn.cursor()
    # 删除该线程的所有存档
    cursor.execute("DELETE FROM checkpoints WHERE thread_id = ?", (thread_id,))
    conn.commit()
    conn.close()
    print(f"已物理重置 thread_id: {thread_id} 的所有记忆")

# 运行它
hard_reset_memory("checkpoints.db", "webster_long_chat")

已物理重置 thread_id: webster_long_chat 的所有记忆


### Summarize messages 消息摘要
消息摘要（Summarization）是管理长对话的“终极方案”。如果说 **Trim** 是剪掉过去，**Remove** 是擦除过去，那么 **Summarization** 就是**浓缩过去**。

这段代码展示了如何使用 `SummarizationMiddleware`（摘要中间件）让 Agent 自动管理自己的“脑容量”。

------

#### 1. 核心参数拆解

在 `create_agent` 的 `middleware` 配置中，摘要中间件有三个关键开关：

- **`model="gpt-4.1-mini"`**：指定一个更便宜、更快的模型专门负责打草稿（写摘要）。没必要用最强的模型来做这种体力活。
- **`trigger=("tokens", 4000)`**：**触发阈值**。当对话累积到 4000 个 Token 时，自动启动“减肥程序”。
- **`keep=("messages", 20)`**：**保留策略**。在压缩过去时，保留最近的 20 条原始对话，确保当下的聊天逻辑不会因为突然断档而显得生硬。

------

#### 2. 它是如何运作的？（隐形的手）

这段代码的神奇之处在于：你虽然连续调用了多次 `invoke`，但摘要的过程是**全自动**的。

1. **静默监控**：中间件会一直盯着你的 `messages` 列表。
2. **自动压缩**：一旦超过 4000 Token，它会把最早的那些消息发给 `mini` 模型，并要求它：“用一段话总结一下这些人刚才聊了什么”。
3. **重组状态**：它会将生成的摘要作为一条新的系统提示（或摘要消息）放在最前面，然后把被总结掉的消息从当前 Context 中移除。

------

#### 3. 代码执行过程分析

1. **第一句：** "hi, my name is bob" —— AI 记住了你的名字。
2. **第二/三句：** 写猫和狗的诗。如果这些诗非常长，导致总 Token 超过了 4000，摘要程序就会启动。
3. **摘要结果：** 历史记录会被压缩成类似：“用户介绍自己叫 Bob，随后 AI 为其创作了关于猫和狗的诗。”
4. **第四句：** "what's my name?" —— 即使原始的 "my name is bob" 已经被从消息列表中删除了，但它已经被**写进了摘要里**。AI 阅读了摘要，依然能准确回答出你叫 Bob。

| **特性**   | **Trim Messages (修剪)** | **Summarization (摘要)**                 |
| ---------- | ------------------------ | ---------------------------------------- |
| **持久性** | 彻底丢失被剪掉的信息。   | 信息的精华被保留，只是细节模糊了。       |
| **逻辑性** | 适合短时上下文。         | 适合长达几天甚至几周的跨度。             |
| **消耗**   | 极低。                   | 稍高（因为需要额外调用一次模型写摘要）。 |

这种“前情提要”式的记忆管理方式，比之前那种“物理切除”看起来更人性化一些。

In [29]:
import operator
from typing import Annotated, Any, TypedDict, cast
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. 定义状态
class CustomAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# 2. 初始化两个不同的模型
# 大脑模型：负责复杂的逻辑和回复
main_model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", 
    model_provider="ollama", 
    temperature=0.1
)

# 秘书模型：负责总结摘要（使用更小的 3b 模型）
summarizer_model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M", 
    model_provider="ollama", 
    temperature=0
)

# 3. 设置持久化存储
memory_context = SqliteSaver.from_conn_string("webster_memory.db")

with memory_context as memory:
    # 4. 创建带有“摘要中间件”的 Agent
    agent = create_agent(
        model=main_model,
        tools=[],
        state_schema=cast(Any, CustomAgentState),
        checkpointer=memory,
        # 核心：添加中间件
        middleware=[
            SummarizationMiddleware(
                model=summarizer_model,      # 使用 3b 模型
                trigger=("messages", 10),    # 当消息达到 10 条时触发摘要
                keep=("messages", 4)         # 摘要后，保留最近的 4 条原始对话，其余压缩
            )
        ],
    )

    # 5. 测试运行
    config = {"configurable": {"thread_id": "webster_long_story_01"}}
    
    # 模拟连续对话
    print("--- 发送第一条信息 ---")
    agent.invoke({"messages": [HumanMessage(content="你好，我叫 Webster。")]}, config)
    
    # 模拟多轮对话（为了触发摘要，可以多聊几句）
    for i in range(10):
        agent.invoke({"messages": [HumanMessage(content=f"这是第 {i} 条关于技术讨论的废话。")]}, config)
        print(f"发送第 {i} 条...")

    # 最后验证 AI 是否还记得你是谁
    print("\n--- 正在测试摘要后的记忆 ---")
    final_response = agent.invoke({"messages": [HumanMessage(content="你还记得我叫什么吗？")]}, config)
    print(f"AI 回复: {final_result['messages'][-1].content}")

--- 发送第一条信息 ---
发送第 0 条...
发送第 1 条...
发送第 2 条...
发送第 3 条...


TypeError: Received unsupported message type for Ollama.

1. 为什么会报错？（深度揭秘）
原因和我们之前折腾 RemoveMessage 时一模一样：

摘要的本质：当摘要触发时，中间件会产生一个特殊的对象。在 LangChain 的某些版本中，它会产生一个 BaseMessage 的变体，或者在消息列表中插入一条包含摘要的特殊记录。

Ollama 的洁癖：Ollama 的 API 极其挑食，它只认识标准的 HumanMessage、AIMessage 和 SystemMessage。

冲突点：中间件在后台生成的“摘要记录”被合并到了 messages 列表中，随后发给了 Ollama。Ollama 一看：“这是什么新型号的消息？我处理不了！”于是直接报错。

##### 解决方式
降级使用 Trim Messages
如果你觉得摘要太麻烦，其实 Trim Messages（配合我们之前说的 approximate）是 Ollama 的最佳拍档。虽然它会丢掉很久以前的信息，但它绝对不会产生让 Ollama 崩溃的特殊消息类型。